## Step 4: Joins and the one-row-per-order table

ML needs one flat table where each row is one order. Items, payments, and products have many rows per order, so we aggregate first, then join.   
`If skipped`: Joining raw tables multiplies rows, so one order appears many times and revenue and labels are wrong.

4A. Aggregate to the order grain


In [1]:
import pandas as pd 
from pathlib import Path

RAW = Path("../data/raw")

files = { "Customers": "olist_customers_dataset.csv",
          "Geolocation": "olist_geolocation_dataset.csv",
          "Items": "olist_order_items_dataset.csv",
          "Payments": "olist_order_payments_dataset.csv",
          "Reviews": "olist_order_reviews_dataset.csv",
          "Orders": "olist_orders_dataset.csv",
          "Products": "olist_products_dataset.csv",
          "Sellers": "olist_sellers_dataset.csv",
          "Category": "product_category_name_translation.csv"
        }
dfs = { name : pd.read_csv(RAW/f) for name , f in files.items() }
for name, df in dfs.items():
    print(f"{name:22} rows={df.shape[0]:>7} cols={df.shape[1]}")
    

Customers              rows=  99441 cols=5
Geolocation            rows=1000163 cols=5
Items                  rows= 112650 cols=7
Payments               rows= 103886 cols=5
Reviews                rows=  99224 cols=7
Orders                 rows=  99441 cols=8
Products               rows=  32951 cols=9
Sellers                rows=   3095 cols=4
Category               rows=     71 cols=2


In [2]:
import pandas as pd, numpy as np
from pathlib import Path

RAW, PROC = Path("../data/raw"), Path("../data/processed")

date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
orders = pd.read_csv(PROC / "orders_clean.csv", parse_dates=date_cols)
orders["estimated_days"] = (orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"]).dt.days

rev  = pd.read_csv(PROC / "reviews_clean.csv")
geo  = pd.read_csv(PROC / "geo_clean.csv")
prod = pd.read_csv(PROC / "products_clean.csv")

items   = pd.read_csv(RAW / "olist_order_items_dataset.csv")
pay     = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
cust    = pd.read_csv(RAW / "olist_customers_dataset.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")



In [3]:
items, pay, cust, sellers = dfs["Items"], dfs["Payments"], dfs["Customers"], dfs["Sellers"]

# items + product info -> one row per order
it = items.merge(
    prod[["product_id", 
          "Category", 
          "product_weight_g", 
          "product_length_cm", 
          "product_height_cm", 
          "product_width_cm"]], 
          on="product_id", 
          how="left")
it["volume_cm3"] = it["product_length_cm"] * it["product_height_cm"] * it["product_width_cm"]

it_agg = it.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    n_sellers=("seller_id", "nunique"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    total_weight_g=("product_weight_g", "sum"),
    total_volume_cm3=("volume_cm3", "sum"),
    seller_id=("seller_id", "first"),
    category=("Category", "first"),
).reset_index()

# payments -> one row per order
pay_agg = pay.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    max_installments=("payment_installments", "max"),
).reset_index()
pay_type = (pay.sort_values("payment_value", ascending=False)
              .drop_duplicates("order_id")[["order_id", "payment_type"]])

4B. Join with a safety check (left joins only)

In [4]:
n0 = len(orders)
base_cols = ["order_id", 
             "customer_id", 
             "order_purchase_timestamp",
             "estimated_days", 
             "is_late", 
             "delivery_days"]

df = orders[base_cols].copy()

# List of merges to perform
merges = [
    (cust[["customer_id", 
           "customer_unique_id", 
           "customer_zip_code_prefix", 
           "customer_state"]],
             "customer_id"),
    (it_agg, "order_id"),
    (pay_agg, "order_id"),
    (pay_type, "order_id"),
    (sellers[["seller_id", 
              "seller_zip_code_prefix", 
              "seller_state"]], "seller_id"),
]
for right, key in merges:
    df = df.merge(right, on=key, how="left")
    assert len(df) == n0, f"Row count changed after joining on {key}!"

print(df.shape)

(96470, 22)


Why left join: An inner join silently drops orders that lack a match, so you lose data without noticing.
Why the assert: It stops the code the moment a join duplicates rows.

4C. Add geography and distance

In [5]:
g = geo.rename(columns={"geolocation_zip_code_prefix": "zip"})
df = (df.merge(g.rename(columns={"zip": "customer_zip_code_prefix", "lat": "cust_lat", "lng": "cust_lng"}),
               on="customer_zip_code_prefix", how="left")
        .merge(g.rename(columns={"zip": "seller_zip_code_prefix", "lat": "sell_lat", "lng": "sell_lng"}),
               on="seller_zip_code_prefix", how="left"))
assert len(df) == n0

import numpy as np
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine(df.sell_lat, df.sell_lng, df.cust_lat, df.cust_lng)
df["same_state"] = (df["customer_state"] == df["seller_state"]).astype(int)

Why distance: Farther delivery means a higher chance of delay. This is likely one of your strongest features.

4D. Time features (known at purchase)

In [6]:
t = df["order_purchase_timestamp"]
df["purchase_month"]   = t.dt.month
df["purchase_weekday"] = t.dt.weekday
df["purchase_hour"]    = t.dt.hour

C:\Users\user\AppData\Local\Temp\ipykernel_10056\1224113992.py:3: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["purchase_weekday"] = t.dt.weekday


4E. Review score for EDA only

In [7]:
df_eda = df.merge(rev[["order_id", "review_score"]], on="order_id", how="left")
print(df_eda)

                               order_id                       customer_id  \
0      e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1      53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2      47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3      949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4      ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   
...                                 ...                               ...   
96465  9c5dedf39a927c1b2549525ed64a053c  39bd1228ee8140590ac3aca26f2dfe00   
96466  63943bddc261676b46f01ca7ac2f7bd8  1fca14ff2861355f6e5f14306ff977a7   
96467  83c1379a015df1e13d02aae0204711ab  1aa71eb042121263aafbe80c1b562c9c   
96468  11c177c8e97725db2631073c19f07b62  b331b74b18dc79bcdf6532d51e1637c1   
96469  66dea50a8b16d9b4dee7af250b4be1a5  edb027a75a1449115f6b43211ae02a24   

      order_purchase_timestamp  estimated_days  is_late  delivery_days  \
0

Leakage warning: A review is written after delivery. It answers question 3 (does lateness hurt reviews?) but is never a model feature. The same applies to delivery_days.


4F. Check and save

In [8]:
print(df.isnull().sum()[lambda s: s > 0])
print(df["is_late"].mean())

df.to_csv("../data/processed/model_table.csv", index=False)
df_eda.to_csv("../data/processed/eda_table.csv", index=False)

payment_value         1
max_installments      1
payment_type          1
cust_lat            264
cust_lng            264
sell_lat            215
sell_lng            215
distance_km         478
dtype: int64
0.08112366538820359


One hidden problem to check

Your null list doesn’t show total_weight_g or total_volume_cm3, but that doesn’t mean they are clean. sum() turns an all-missing group into 0, which hides the missing value as a fake weight. Check:

In [9]:
print((df["total_weight_g"] == 0).sum(), (df["total_volume_cm3"] == 0).sum())

22 16


If the count is more than 0, those are really missing. Fix it with:


In [10]:
df.loc[df["total_weight_g"] == 0, ["total_weight_g", "total_volume_cm3"]] = np.nan

Still missing from your Step 4 results

In [11]:
print((df["n_sellers"] > 1).sum())
print(df.groupby("is_late")["distance_km"].mean())

1275
is_late
0    588.504580
1    738.836414
Name: distance_km, dtype: float64


Save and commit


In [12]:
df.to_csv("../data/processed/model_table.csv", index=False)

`powershell`  
git add .  
git commit -m "Step 4: order-level feature table with distance"  
git push